# Cabouy - consolidation des chroniques

Trois sondes : **CTD** (Diver autonome : niveau, conductivité, température),
**TROLL** (Aqua TROLL : conductivité, température, turbidité, O2, chlorophylle,
**pas de niveau**), **OTT** (la sonde CTD de la centrale : niveau, conductivité,
température).

La centrale rapatrie aussi les voies du TROLL (`C2`, `T2`, `Turbi`, `O2`,
`Chlorophyl`) : c'est le **même capteur** que les exports VuSitu, un second chemin
d'acquisition, pas une quatrième sonde. Les deux sont réunis en cellule 8, sans
recalage.

Pour chaque grandeur, une liste `PERIODES_*` dit **quelle sonde est prioritaire sur
quelle période**. Elle est écrite en dur, juste au-dessus du graphe de correction.
La cellule 9 la propose une fois, à partir de la disponibilité réelle des sondes.

## 1. Imports

In [ ]:
import os
import re
import unicodedata
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## 2. Chemins d'accès

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes\CTD")
VUSITU_PATH = os.path.join(BASE, r"Données brutes\TROLL")
OTT_PATH    = os.path.join(BASE, r"Données brutes\OTT")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"
PLUIE_PATH  = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Saint Sauveur\Gaetan\Données brutes\Pluie_BV_Ouysse.csv"

OLDDATA_PATH     = os.path.join(BASE, "Cabouy_consolide_OLD.xlsx")
UTC_CTD_PATH     = os.path.join(BASE, "UTC_CTD.xlsx")
UTC_TROLL_PATH   = os.path.join(BASE, "UTC_Troll.xlsx")
PUNCTUAL_NIVEAU  = os.path.join(BASE, "punctual_measurements.xlsx")
PUNCTUAL_CONDUCT = os.path.join(BASE, "punctual_measurements_conducti.xlsx")
SORTIE_CONSOLIDE = os.path.join(BASE, "Cabouy_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, "Cabouy_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, "Graphes.svg")

PREFIXE_CTD = "Cabouy"
BARO_COL    = "Patm Ouysse Calès [hPa]"
PAS         = "1h"

## 3. Fonctions de lecture

Les pièges de format : en-tête Diver à une ligne variable et pied `END OF DATA`,
guillemets et numéros de série VuSitu, sentinelle `-99999` de la centrale, virgules
décimales, encodages. Un fichier absent de la table UTC est ignoré, avec son nom :
c'est la table qui se corrige.

In [ ]:
def _sans_accents(t):
    d = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in d if not unicodedata.combining(c)).lower()


def _lire_lignes(chemin):
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            with open(chemin, "r", encoding=enc) as f:
                return f.read().splitlines(), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Aucun encodage ne convient pour {chemin}")


def _en_datetime(serie):
    """Garde le format qui convertit le plus de lignes."""
    txt = serie.astype("string").str.strip()
    meilleur, n_ok = None, -1
    for fmt in ("%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M"):
        e = pd.to_datetime(txt, format=fmt, errors="coerce")
        if e.notna().sum() > n_ok:
            meilleur, n_ok = e, e.notna().sum()
    if n_ok < len(txt):                      # format inconnu : lecture libre
        libre = pd.to_datetime(txt, errors="coerce", dayfirst=True)
        meilleur = libre if libre.notna().sum() > n_ok else meilleur
    return meilleur


def fichiers(path, motif):
    """Fichiers du dossier contenant `motif`, triés par numéro."""
    noms = [f for f in os.listdir(path)
            if motif.lower() in f.lower() and f.lower().endswith((".csv", ".txt", ".mon"))]
    return sorted(noms, key=lambda n: (int(re.findall(r"\d+", n)[0]) if re.findall(r"\d+", n) else 10 ** 9, n))


def lire_CTD(nom, path=CTD_PATH):
    """Export Diver : en-tête cherchée par contenu, pied END OF DATA reconnu,
    conductivité convertie d'après l'unité entre crochets."""
    chemin = os.path.join(path, nom)
    lignes, encodage = _lire_lignes(chemin)
    entete = next((i for i, l in enumerate(lignes[:200])
                   if _sans_accents(l).lstrip("\ufeff").startswith("date/time")), None)
    if entete is None:
        raise ValueError(f"En-tête 'Date/time' introuvable dans {nom}")

    df = pd.read_csv(chemin, sep=";", encoding=encodage, skiprows=entete, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]
    col = df.columns[0]
    brut = df[col].astype("string")
    fin = brut.map(lambda v: pd.notna(v) and "end of data" in _sans_accents(v)).fillna(False)
    df = df.loc[~(fin | brut.isna() | (brut.str.strip() == ""))].copy()

    df["Date/time"] = _en_datetime(df[col]).dt.round(PAS)
    for c in df.columns:
        if c not in ("Date/time", col):
            df[c] = pd.to_numeric(df[c].astype("string").str.strip()
                                  .str.replace(",", ".", regex=False), errors="coerce")
    for c in list(df.columns):
        if "cond" in _sans_accents(c):
            u = re.search(r"\[([^\]]*)\]", c)
            df[c] = df[c] * (1000.0 if u and _sans_accents(u.group(1)).startswith("ms/cm") else 1.0)
            df = df.rename(columns={c: "Cond_(µS/cm)"})
            break
    return df.loc[df["Date/time"].notna()].sort_values("Date/time")


#: Libellés VuSitu (sans numéro de série, sans accent) vers les noms du projet.
NOMS_TROLL = {
    "conductivite specifique (us/cm)":        "Cond_Troll_(µS/cm)",
    "temperature (c)":                        "température_Troll_(°C)",
    "turbidite (ntu)":                        "Turbidity_Troll_(NTU)",
    "concentration rdo (mg/l)":               "O2_Troll_(mg/l)",
    "saturation rdo (%sat)":                  "O2 (%Sat)",
    "fluorescence de chlorophylle-a (rfu)":   "FluorescenceChloro_a_Troll_(RFU)",
    "concentration de chlorophylle-a (ug/l)": "ConcentrationChloro_a_(µg/l)",
}


def lire_VuSitu(nom, path=VUSITU_PATH):
    """Export VuSitu : guillemets retirés, numéro de série retiré par regex."""
    lignes, _ = _lire_lignes(os.path.join(path, nom))
    df = pd.read_csv(StringIO("\n".join(l.replace('"', "") for l in lignes)), sep=",")
    cle = lambda c: (_sans_accents(re.sub(r"\s*\(\d{4,}\)\s*$", "", str(c)).strip())
                     .replace("\u03bc", "u").replace("\u00b5", "u").replace("\u00b0", ""))
    col = next((c for c in df.columns if "date" in _sans_accents(c)), df.columns[0])
    df["DATE"] = _en_datetime(df[col]).dt.round(PAS)
    return df.drop(columns=[col]).rename(
        columns={c: NOMS_TROLL[cle(c)] for c in df.columns if cle(c) in NOMS_TROLL})


#: Voies de la centrale. level, C1, T1 = SA sonde CTD.
#: C2, T2, Turbi, O2, Chlorophyl = le TROLL rapatrié, donc un doublon.
NOMS_OTT = {"level": "Niveau_CTDOTT_(cm)",
            "c1": "Cond_CTDOTT_(µS/cm)",   "t1": "Temp_CTDOTT_(°C)",
            "c2": "Cond_TrollOTT_(µS/cm)", "t2": "Temp_TrollOTT_(°C)",
            "turbi": "Turbidity_TrollOTT_(NTU)", "o2": "O2_TrollOTT_(mg/l)",
            "chlorophyl": "FluorescenceChloro_a_TrollOTT_(RFU)"}


def lire_OTT(nom, path=OTT_PATH):
    """Centrale, déjà en UTC : -99999 = absence, horodatages en double départagés."""
    chemin = os.path.join(path, nom)
    _, encodage = _lire_lignes(chemin)
    df = pd.read_csv(chemin, sep=";", encoding=encodage, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]
    col = df.columns[0]
    df["DATE"] = _en_datetime(df[col]).dt.round(PAS)
    df = df.drop(columns=[col])
    for c in df.columns:
        if c != "DATE":
            df[c] = pd.to_numeric(df[c].astype("string").str.strip()
                                  .str.replace(",", ".", regex=False), errors="coerce")
            df.loc[df[c].isin([-99999, -9999, 9999]), c] = np.nan
    df = df.rename(columns={c: NOMS_OTT[c.lower()] for c in df.columns if c.lower() in NOMS_OTT})
    return df.dropna(subset=["DATE"]).groupby("DATE", as_index=False).median(numeric_only=True)


def en_utc(df, nom, metadata, col_date="Date/time", col_utc="UTC Fichier"):
    """Ramène les horodatages en UTC. Correspondance EXACTE sur le nom du
    fichier : sinon la campagne est ignoree et le message la nomme."""
    ligne = metadata.loc[metadata["Nom fichier"] == nom, col_utc]
    if ligne.empty:
        raise ValueError(f"'{nom}' absent de la colonne 'Nom fichier'. "
                         f"À corriger dans la table UTC.")
    v = ligne.values[0]
    m = re.search(r"([+-]?\d+(?:[.,]\d+)?)", str(v))
    decalage = float(v) if isinstance(v, (int, float, np.number)) and pd.notna(v) else (
        float(m.group(1).replace(",", ".")) if m else 0.0)
    df = df.copy()
    df[col_date] = df[col_date] - pd.Timedelta(hours=decalage)
    return df, decalage


#: Gamme physique par mot-clé de nom de colonne. Pas de seuil sur le NIVEAU :
#: il n'a pas d'origine absolue tant qu'il n'est pas calé.
GAMMES = {"cond": (30, 5000), "temp": (-2, 30), "turbid": (0, 4000),
          "o2": (0, 25), "chloro": (0, 500)}


def appliquer_gammes(df):
    """Met à NaN ce qui est physiquement impossible, voie par voie."""
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        for cle, (mini, maxi) in GAMMES.items():
            if cle in _sans_accents(col):
                hors = (df[col] < mini) | (df[col] > maxi)
                if hors.any():
                    print(f"  {col:36s} {int(hors.sum()):6d} hors [{mini}, {maxi}]")
                    df.loc[hors, col] = np.nan
                break
    return df

### Fonctions de correction

`decaler` porte le choix du sens : `aval` pour une marche réelle (capteur déplacé),
`amont` pour ramener l'historique sur la référence actuelle, `tout` pour un calage
global. On met à **NaN**, jamais de ligne supprimée.

`choisir_sondes` découpe la chronique en périodes : à chaque pas, la première sonde
disponible de `ORDRE`, sauf sur les périodes d'exception où la sonde nommée passe en
tête. On ne change pas de sonde pour boucher un trou de moins de 12 h : c'est
l'interpolation qui s'en charge.

`fusionner` enchaîne ces périodes : chacune n'utilise **que** sa sonde, et à chaque
changement la nouvelle est recalée sur la précédente.

In [ ]:
def graphe(traces, titre="", ylab="", points=None, col_point=None):
    """`traces` = liste de (série, nom, couleur). Scattergl : une chronique
    horaire pluriannuelle s'affiche sans saturer le navigateur."""
    fig = go.Figure()
    for serie, nom, couleur in traces:
        fig.add_trace(go.Scattergl(x=serie.index, y=serie, mode="lines", name=nom,
                                   line=dict(color=couleur, width=1.3)))
    if points is not None and col_point in points.columns:
        corr = points.get("Correction", pd.Series("Non", index=points.index))
        fig.add_trace(go.Scattergl(
            x=points["Datetime"], y=points[col_point], mode="markers", name="points de contrôle",
            marker=dict(symbol="x", size=10,
                        color=["red" if str(v).strip() == "Oui" else "royalblue" for v in corr])))
    fig.update_layout(title=titre, xaxis_title="Date", yaxis_title=ylab,
                      template="plotly_white", hovermode="x unified")
    fig.show()          # pas de `return fig` : sinon Jupyter réaffiche la figure


#: Couleur de chaque sonde, la même sur tous les graphes.
COULEURS = {"OTT": "#1f77b4", "TROLL": "#d62728", "CTD": "#2ca02c"}


def decaler(serie, date, valeur, sens="tout"):
    """Ajoute `valeur` a toute la serie, a l'aval de `date` (incluse) ou a
    l'amont (strictement avant)."""
    if sens == "tout":
        return serie + valeur
    date = pd.to_datetime(date)
    m = np.asarray(serie.index >= date if sens == "aval" else serie.index < date)
    return serie.where(~m, serie + valeur)


def ecarter(serie, periodes):
    """Passe a NaN les periodes (debut, fin, motif) et le dit."""
    for debut, fin, motif in periodes:
        debut, fin = sorted([pd.to_datetime(debut), pd.to_datetime(fin)])
        m = np.asarray((serie.index >= debut) & (serie.index <= fin))
        print(f"  {debut:%d/%m/%Y %H:%M} - {fin:%d/%m/%Y %H:%M} : "
              f"{int((m & serie.notna().to_numpy()).sum())} pas écartés ({motif})")
        serie = serie.mask(m)
    return serie


def caler(serie, points, col_valeur, tolerance_h=1):
    """Recale la serie sur les mesures ponctuelles marquees Oui, en cascade
    vers l'aval. Chaque point est trace, applique ou non."""
    for _, l in points.dropna(subset=["Datetime"]).sort_values("Datetime").iterrows():
        date, cible = l["Datetime"], l.get(col_valeur)
        if pd.isna(cible) or str(l.get("Correction", "Non")).strip() != "Oui":
            continue
        mesures = serie.dropna()
        i = mesures.index[np.abs((mesures.index - date).to_numpy()).argmin()] if len(mesures) else None
        if i is None or abs((i - date).total_seconds()) > tolerance_h * 3600:
            print(f"  {date:%d/%m/%Y %H:%M} : ignoré, pas de mesure à moins de {tolerance_h} h")
            continue
        d = float(cible) - float(serie.loc[i])
        print(f"  {date:%d/%m/%Y %H:%M} : {float(serie.loc[i]):.1f} vers {float(cible):.1f}, "
              f"décalage {d:+.2f} appliqué vers l'aval")
        serie = decaler(serie, date, d, "aval")
    return serie


def raccorder(ancienne, nouvelle, voisinage, transition, sonde, unite="", trou_max_h=12):
    """Decalage a ajouter a `nouvelle` pour qu'elle prolonge `ancienne` au
    changement de periode.

    Mediane des ecarts sur le recouvrement des deux sondes AU VOISINAGE de la
    transition : un recalage sur toute leur periode commune melangerait des
    situations differentes. A defaut de recouvrement, raccord bout a bout si le
    trou est court ; au-dela de `trou_max_h` il n'y a rien pour caler les deux
    sondes l'une sur l'autre, donc on ne recale pas et on laisse le trou plutot
    que d'inventer un raccord.
    """
    commun = voisinage & ancienne.notna().to_numpy() & nouvelle.notna().to_numpy()
    if commun.any():
        d = float((ancienne[commun] - nouvelle[commun]).median())
        note = f"recouvrement de {int(commun.sum())} pas"
    else:
        a = ancienne[ancienne.index <= transition].dropna()
        b = nouvelle[nouvelle.index >= transition].dropna()
        trou = ((b.index[0] - a.index[-1]).total_seconds() / 3600
                if len(a) and len(b) else np.inf)
        if trou <= trou_max_h:
            d = float(a.iloc[-1] - b.iloc[0])
            note = f"trou de {trou:.0f} h, raccord bout à bout"
        else:
            d, note = 0.0, f"trou de {trou:.0f} h : aucun recalage possible"
    print(f"  {sonde:6s} recalée de {d:+.2f} {unite}  ({note})")
    return d


def choisir_sondes(voies, ordre, exceptions=(), duree_mini_h=12):
    """Decoupe la chronique en periodes (debut, fin, sonde).

    A chaque pas, la premiere sonde disponible de `ordre`. Sur une periode
    d'exception (debut, fin, sonde), la sonde nommee passe en tete la ou elle
    mesure. Un bloc plus court que `duree_mini_h` est absorbe par le
    precedent : on ne change pas de sonde pour boucher un trou de quelques
    heures, c'est l'interpolation qui s'en charge.
    """
    index = next(iter(voies.values())).index
    choix = pd.Series(pd.NA, index=index, dtype="object")
    for sonde in [s for s in ordre if s in voies] + [s for s in voies if s not in ordre]:
        choix = choix.where(choix.notna() | voies[sonde].isna(), sonde)
    for debut, fin, sonde in exceptions:
        p = np.asarray((index >= pd.to_datetime(debut)) & (index <= pd.to_datetime(fin)))
        choix = choix.where(~(p & voies[sonde].notna().to_numpy()), sonde)

    choix, blocs = choix.dropna(), []
    mini = pd.Timedelta(hours=duree_mini_h)
    for _, g in choix.groupby((choix != choix.shift()).cumsum()):
        if blocs and (g.iloc[0] == blocs[-1][2] or g.index[-1] - g.index[0] < mini):
            blocs[-1][1] = g.index[-1]
        else:
            blocs.append([g.index[0], g.index[-1], g.iloc[0]])
    return [tuple(b) for b in blocs]


def fusionner(voies, periodes, unite="", trou_max_h=12, fenetre_j=30):
    """Chronique d'une grandeur. `voies` = {sonde: serie}.

    Chaque periode n'utilise QUE sa sonde : aucune autre ne vient combler ses
    lacunes. A chaque changement de sonde, la nouvelle est recalee sur la
    precedente, en cascade depuis la premiere periode, qui fixe le zero. Le
    recalage se mesure sur `fenetre_j` jours de part et d'autre de la
    transition.
    """
    index = next(iter(voies.values())).index
    valeur = pd.Series(np.nan, index=index)
    source = pd.Series(pd.NA, index=index, dtype="object")
    decalage, precedente = 0.0, None
    for debut, fin, sonde in periodes:
        transition = pd.to_datetime(debut)
        p = np.asarray((index >= transition) & (index <= pd.to_datetime(fin)))
        serie = voies[sonde]
        print(f"  {transition:%d/%m/%Y %H:%M} - {pd.to_datetime(fin):%d/%m/%Y %H:%M}  {sonde}")
        if precedente is not None and sonde != precedente:
            fen = pd.Timedelta(days=fenetre_j)
            voisinage = np.asarray((index >= transition - fen) & (index <= transition + fen))
            decalage = raccorder(voies[precedente] + decalage, serie, voisinage,
                                 transition, sonde, unite, trou_max_h)
        pris = p & serie.notna().to_numpy()
        valeur[pris], source[pris] = serie[pris] + decalage, sonde
        precedente = sonde
    return valeur, source

## 4. CTD : lecture, UTC et compensation barométrique

`1 hPa = 1.019716 cmH2O`. Mettre `HPA_EN_CMH2O = 1.0` redonne l'ancienne formule.

In [ ]:
HPA_EN_CMH2O = 1.019716

metadata = pd.read_excel(UTC_CTD_PATH)
baro = pd.read_excel(BARO_PATH)[["DATE", BARO_COL]]
baro["DATE"] = pd.to_datetime(baro["DATE"], errors="coerce")
baro = baro.dropna(subset=["DATE"]).drop_duplicates("DATE")

morceaux = []
for nom in fichiers(CTD_PATH, PREFIXE_CTD):
    try:
        CTD, decalage = en_utc(lire_CTD(nom), nom, metadata)
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    m = pd.merge(CTD, baro, left_on="Date/time", right_on="DATE", how="left")
    m["Niveau_(cm)"] = m["Pression[cmH2O]"] - m[BARO_COL] * HPA_EN_CMH2O
    m = m.rename(columns={"Température[°C]": "Temp_(°C)"})
    morceaux.append(m[["Date/time", "Niveau_(cm)", "Cond_(µS/cm)", "Temp_(°C)"]])
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(CTD)} lignes)")

merge_ctd_df = pd.concat(morceaux, ignore_index=True).sort_values("Date/time", kind="stable")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(morceaux)} campagne(s), {len(merge_ctd_df)} enregistrements en UTC.")

## 5. Raccordement à l'ancienne chronique

L'ancien fichier consolidé et les campagnes sont la **même sonde CTD**, séparées par
un trou d'exploitation. Le décalage est mesuré à la jonction, comme la V2, puis
ajouté aux campagnes. `RACCORD_FIGE` sert à figer une valeur quand elle convient :
`RACCORD_FIGE = {"Niveau": 76.86}` fige le niveau et laisse la conductivité calculée.

In [ ]:
RACCORD_FIGE = {}          # ex. {"Niveau": 76.86, "Conductivité": 5.5}

olddata_df = pd.read_excel(OLDDATA_PATH)
olddata_df["DATE"] = pd.to_datetime(olddata_df["DATE"], errors="coerce")
olddata_df = olddata_df.dropna(subset=["DATE"]).sort_values("DATE")

ancien, nouveau = olddata_df.set_index("DATE"), merge_ctd_df.set_index("DATE")
grandeurs = [("Niveau", "Niveau_(cm)", "Niveau_(cm)", "cm"),
             ("Conductivité", "Cond_CTD_(µS/cm)", "Cond_(µS/cm)", "µS/cm")]
for nom, col_a, col_n, unite in grandeurs:
    a, n = ancien[col_a].dropna(), nouveau[col_n].dropna()
    jonction = float(a.iloc[-1] - n.iloc[0]) if len(a) and len(n) else 0.0
    d = float(RACCORD_FIGE.get(nom, jonction))
    origine = "figé" if nom in RACCORD_FIGE else "mesuré à la jonction"
    print(f"{nom:13s} : {d:+9.2f} {unite:6s} {origine}"
          + (f" (jonction : {jonction:+.2f})" if nom in RACCORD_FIGE else
             f"  {a.index[-1]:%d/%m/%Y %H:%M} vers {n.index[0]:%d/%m/%Y %H:%M}"))
    merge_ctd_df[col_n] = merge_ctd_df[col_n] + d

bord = olddata_df["DATE"].max()
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, (nom, col_a, col_n, unite) in zip(axes, grandeurs):
    ax.plot(olddata_df["DATE"], olddata_df[col_a], color="green", label="ancienne chronique")
    ax.plot(merge_ctd_df["DATE"], merge_ctd_df[col_n], color="blue", label="campagnes raccordées")
    ax.set_ylabel(f"{nom} ({unite})")
    ax.legend(loc="upper left")
axes[0].set_xlim(bord - pd.Timedelta(days=7), bord + pd.Timedelta(days=7))
axes[0].set_title("Raccordement, 7 jours de part et d'autre de la jonction")
plt.tight_layout()
plt.show()

## 6. TROLL : exports VuSitu

In [ ]:
metadata_troll = pd.read_excel(UTC_TROLL_PATH, sheet_name=0)

morceaux = []
for nom in fichiers(VUSITU_PATH, "VuSitu"):
    try:
        df_v, decalage = en_utc(lire_VuSitu(nom), nom, metadata_troll, col_date="DATE")
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    morceaux.append(df_v)
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(df_v)} lignes)")

merge_troll_df = (pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"])
                  .sort_values("DATE", kind="stable"))
print(f"\n{len(morceaux)} fichier(s), {len(merge_troll_df)} enregistrements en UTC.")

## 7. Centrale OTT

In [ ]:
merge_ott_df = (pd.concat([lire_OTT(nom) for nom in fichiers(OTT_PATH, "")], ignore_index=True)
                .groupby("DATE", as_index=False).median(numeric_only=True).sort_values("DATE"))
print(f"{len(merge_ott_df)} pas de temps, du {merge_ott_df['DATE'].min():%d/%m/%Y} "
      f"au {merge_ott_df['DATE'].max():%d/%m/%Y}")

## 8. Assemblage : une colonne par sonde

Deux dédoublonnages différents, à ne pas confondre.

**Au sein d'une même lignée** (ancien fichier + campagnes CTD qui se recouvrent),
sur un horodatage présent deux fois, la **première valeur gagne**, comme la V2 :
c'est la campagne la plus ancienne, déjà relue et validée.

**Entre les deux chemins du TROLL**, l'export **VuSitu direct est prioritaire** ; la
voie rapatriée par la centrale ne sert qu'à combler ses trous, sans recalage. C'est
le même capteur, la centrale n'en est qu'un enregistrement à une autre résolution.

In [ ]:
COLONNES_CTD   = ["Niveau_CTD_(cm)", "Cond_CTD_(µS/cm)", "Temp _CTD(°C)"]
COLONNES_TROLL = ["Cond_Troll_(µS/cm)", "température_Troll_(°C)", "Turbidity_Troll_(NTU)",
                  "O2_Troll_(mg/l)", "O2 (%Sat)", "FluorescenceChloro_a_Troll_(RFU)",
                  "ConcentrationChloro_a_(µg/l)"]
#: (voie directe, même voie rapatriée par la centrale)
DOUBLONS = [("Cond_Troll_(µS/cm)", "Cond_TrollOTT_(µS/cm)"),
            ("température_Troll_(°C)", "Temp_TrollOTT_(°C)"),
            ("Turbidity_Troll_(NTU)", "Turbidity_TrollOTT_(NTU)"),
            ("O2_Troll_(mg/l)", "O2_TrollOTT_(mg/l)"),
            ("FluorescenceChloro_a_Troll_(RFU)", "FluorescenceChloro_a_TrollOTT_(RFU)")]


def empiler(morceaux, colonnes):
    pile = pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"])
    pile = pile[["DATE"] + [c for c in colonnes if c in pile.columns]]
    return (pile.sort_values("DATE", kind="stable")
            .drop_duplicates("DATE", keep="first").set_index("DATE"))


piles = [
    empiler([olddata_df.rename(columns={"Niveau_(cm)": "Niveau_CTD_(cm)"}),
             merge_ctd_df.rename(columns={"Niveau_(cm)": "Niveau_CTD_(cm)",
                                          "Cond_(µS/cm)": "Cond_CTD_(µS/cm)",
                                          "Temp_(°C)": "Temp _CTD(°C)"})], COLONNES_CTD),
    empiler([olddata_df, merge_troll_df], COLONNES_TROLL),
    empiler([merge_ott_df], list(NOMS_OTT.values())),
]

grille = pd.date_range(min(p.index.min() for p in piles),
                       max(p.index.max() for p in piles), freq=PAS, name="DATE")
full_data = pd.DataFrame(index=grille)
for pile in piles:
    for col in pile.columns:
        full_data[col] = pile[col].reindex(grille)
print(f"{len(full_data)} pas horaires, du {grille.min():%d/%m/%Y} au {grille.max():%d/%m/%Y}")

print("Hors gamme physique :")
full_data = appliquer_gammes(full_data)

print("Trous du TROLL direct comblés par la centrale :")
for direct, relais in DOUBLONS:
    trou = (full_data[direct].isna() & full_data[relais].notna()).to_numpy()
    full_data[direct] = full_data[direct].where(~trou, full_data[relais])
    print(f"  {direct:34s} {int(trou.sum()):6d} pas")

#: Les sondes de chaque grandeur : {grandeur: {sonde: colonne}}.
SONDES = {
    "Niveau_(cm)":        {"OTT": "Niveau_CTDOTT_(cm)", "CTD": "Niveau_CTD_(cm)"},
    "Conductivité":       {"OTT": "Cond_CTDOTT_(µS/cm)", "TROLL": "Cond_Troll_(µS/cm)",
                           "CTD": "Cond_CTD_(µS/cm)"},
    "Température":        {"OTT": "Temp_CTDOTT_(°C)", "TROLL": "température_Troll_(°C)",
                           "CTD": "Temp _CTD(°C)"},
    "Turbidité_(NTU)":    {"TROLL": "Turbidity_Troll_(NTU)"},
    "O2_(mg/l)":          {"TROLL": "O2_Troll_(mg/l)"},
    "Chlorophylle_(RFU)": {"TROLL": "FluorescenceChloro_a_Troll_(RFU)"},
}
PARAMETRES = list(SONDES)

#: Ordre de preference automatique : a chaque pas, la premiere sonde qui mesure.
#: Chaque cellule de correction peut imposer une autre sonde sur une periode.
ORDRE = ["OTT", "TROLL", "CTD"]

#: Voies écartées au jugement, AVANT la fusion : (début, fin, colonne, motif).
#: Une sonde qui dérive est écartée, une autre prend le relais.
VOIES_ECARTEES = [
    ("2023-09-05 13:00", "2023-10-21 22:00", "Temp _CTD(°C)", "dérive sonde CTD"),
    ("2021-04-09 14:00", "2021-06-03 12:00", "O2_Troll_(mg/l)", "capteur RDO défaillant"),
]
print("Voies écartées au jugement :")
for debut, fin, col, motif in VOIES_ECARTEES:
    full_data[col] = ecarter(full_data[col], [(debut, fin, f"{col} : {motif}")])

# Instantané des voies brutes. Les cellules de correction en repartent, jamais
# de la colonne qu'elles écrivent : relancer une cellule cumulerait les décalages.
BRUT = full_data.copy()

#: Ajustements manuels d'une sonde, hors points de contrôle :
#: (date d'ancrage, sonde, grandeur, décalage, sens).
#: sens = "amont" (avant la date) | "aval" (à partir de la date) | "tout".
#: Le 06/12/2024 16:00 UTC la centrale lisait 107 cm à l'échelle et la CTD
#: 76.86 cm de moins ; l'échelle a été déplacée en juin 2024 mais le capteur
#: n'a pas bougé, donc le calage porte sur le PASSÉ de cette lecture.
CALAGES_SONDE = [
    ("2024-12-06 16:00", "CTD", "Niveau_(cm)", 76.86, "amont"),
]


def voies_calees(grandeur):
    """Les series des sondes d'une grandeur, ajustements manuels appliques."""
    series = {s: BRUT[c] for s, c in SONDES[grandeur].items()}
    for date, sonde, cible, valeur, sens in CALAGES_SONDE:
        if cible == grandeur and sonde in series:
            series[sonde] = decaler(series[sonde], date, valeur, sens)
            print(f"  {sonde} : {valeur:+.2f} en {sens} du "
                  f"{pd.to_datetime(date):%d/%m/%Y %H:%M}")
    return series
full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\nDétail capteur par capteur : {SORTIE_CONSOLIDE}")

## 9. Comparaison des sources

Les sondes brutes, superposées : c'est ici qu'on juge laquelle est la plus fiable et
sur quelle période, avant d'écrire une exception dans les cellules suivantes.

In [ ]:
PARAMETRE = "Conductivité"   # "Niveau_(cm)", "Température", "Turbidité_(NTU)", "O2_(mg/l)", "Chlorophylle_(RFU)"

graphe([(BRUT[col], f"sonde {sonde}", COULEURS[sonde])
        for sonde, col in SONDES[PARAMETRE].items()],
       titre=f"{PARAMETRE} : les sondes disponibles", ylab=PARAMETRE)

## 10. Niveau

La sonde est choisie automatiquement, dans l'ordre `ORDRE` déclaré en cellule 8.
`EXCEPTIONS_NIVEAU` sert à imposer une autre sonde sur une période précise, quand le
graphe montre que le choix automatique n'est pas le bon. Les périodes retenues sont
affichées, avec le recalage appliqué à chaque changement de sonde.

Le calage de la CTD sur l'échelle est déclaré en cellule 8, dans `CALAGES_SONDE`.

In [ ]:
#: (début, fin, sonde imposée) : sort du choix automatique sur cette période.
EXCEPTIONS_NIVEAU = [
    # ("2023-07-24 01:00", "2023-09-05 12:00", "CTD"),
]
PERIODES_ECARTEES_NIVEAU = [
    ("2019-10-14 17:00", "2020-02-25 17:00", "sonde déplacée"),
]

points_niveau = pd.read_excel(PUNCTUAL_NIVEAU)
points_niveau["Datetime"] = pd.to_datetime(points_niveau["Jour"], dayfirst=True, errors="coerce")

print("Calages de sonde :")
voies = voies_calees("Niveau_(cm)")
print("Périodes retenues :")
avant, source = fusionner(voies, choisir_sondes(voies, ORDRE, EXCEPTIONS_NIVEAU), "cm")

print("Périodes écartées :")
niveau = ecarter(avant, PERIODES_ECARTEES_NIVEAU)
print("Points de contrôle :")
niveau = caler(niveau, points_niveau, "Hauteur (cm)")
full_data["Niveau_(cm)"], full_data["Niveau_(cm)_source"] = niveau, source
print("Pas de temps par sonde :", source.value_counts().to_dict())

# Les sondes sont tracées telles qu'elles entrent dans la fusion, calages
# manuels compris : les écarts visibles sont ceux dont il faut tenir compte
# pour écrire une exception ci-dessus.
graphe([(serie, f"sonde {sonde}", COULEURS[sonde]) for sonde, serie in voies.items()]
       + [(avant, "fusion, avant correction", "lightgrey"),
          (niveau, "chronique corrigée", "black")],
       titre="Niveau", ylab="Niveau (cm)", points=points_niveau, col_point="Hauteur (cm)")

## 11. Conductivité

In [ ]:
#: (début, fin, sonde imposée) : sort du choix automatique sur cette période.
EXCEPTIONS_COND = [
    # ("2023-07-24 01:00", "2023-09-05 12:00", "CTD"),
]
PERIODES_ECARTEES_COND = [
    # ("2021-01-01 00:00", "2021-01-31 00:00", "motif"),
]
FENETRE_IQR, K_IQR = "800h", 1.5       # filtre appliqué à toute la chronique

points_cond = pd.read_excel(PUNCTUAL_CONDUCT)
points_cond["Datetime"] = pd.to_datetime(points_cond["Jour"], dayfirst=True, errors="coerce")

print("Calages de sonde :")
voies = voies_calees("Conductivité")
print("Périodes retenues :")
avant, source = fusionner(voies, choisir_sondes(voies, ORDRE, EXCEPTIONS_COND), "µS/cm")

print("Périodes écartées :")
cond = ecarter(avant, PERIODES_ECARTEES_COND)
print("Points de contrôle :")
cond = caler(cond, points_cond, "Conductivité")

r = cond.rolling(FENETRE_IQR, center=True, min_periods=8)
q1, q3 = r.quantile(0.25), r.quantile(0.75)
hors = ((cond < q1 - K_IQR * (q3 - q1)) | (cond > q3 + K_IQR * (q3 - q1))).fillna(False)
cond = cond.mask(hors)
print(f"Filtre IQR ({FENETRE_IQR}, k={K_IQR}) : {int(hors.sum())} valeurs écartées")

full_data["Conductivité"], full_data["Conductivité_source"] = cond, source
full_data["Conductivité_Moyenne_Mobile"] = cond.rolling("6h", center=True).mean()
print("Pas de temps par sonde :", source.value_counts().to_dict())

graphe([(serie, f"sonde {sonde}", COULEURS[sonde]) for sonde, serie in voies.items()]
       + [(avant, "fusion, avant correction", "lightgrey"),
          (cond, "chronique corrigée", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)",
       points=points_cond, col_point="Conductivité")

## 12. Température et autres paramètres

Choix automatique, pas de correction. Les voies défaillantes ont déjà été écartées
en cellule 8.

In [ ]:
#: {grandeur: [(début, fin, sonde imposée)]} pour sortir du choix automatique.
EXCEPTIONS = {}

for grandeur in ["Température", "Turbidité_(NTU)", "O2_(mg/l)", "Chlorophylle_(RFU)"]:
    print(f"{grandeur} :")
    voies = voies_calees(grandeur)
    full_data[grandeur], full_data[f"{grandeur}_source"] = fusionner(
        voies, choisir_sondes(voies, ORDRE, EXCEPTIONS.get(grandeur, [])))
    print(f"  {full_data[grandeur].notna().sum()} pas  "
          f"{full_data[f'{grandeur}_source'].value_counts().to_dict()}")

## 13. Cote NGF, interpolation et statuts

`cote = 107.6158 + Niveau_(cm) / 100`, le zéro de l'échelle étant à 107.6158 m NGF.

La V2 écrivait `ngf - (ngf - h) / 100`, qui se simplifie en `106.5396 + h / 100` :
elle plaçait donc le zéro **1.076 m trop bas**. Les cotes NGF sortant d'ici sont
supérieures de 1.0762 m à celles de la V2.

Les lacunes de moins de 12 h sont comblées. `Statut_<grandeur>` dit si la valeur est
mesurée, interpolée ou manquante. La cote se recalcule depuis le niveau interpolé.

In [ ]:
NIVEAU_NGF_CABOUY = 107.6158       # cote du zéro de l'échelle
MAX_TROU_H = 12

def cote_ngf(niveau_cm):
    return NIVEAU_NGF_CABOUY + niveau_cm / 100

max_pas = int(pd.Timedelta(f"{MAX_TROU_H}h") / pd.Timedelta(PAS))
for col in PARAMETRES:
    origine = full_data[col]
    manquant = origine.isna().to_numpy()
    groupe = np.cumsum(np.r_[True, manquant[1:] != manquant[:-1]])
    tailles = pd.Series(groupe).groupby(groupe).transform("size").to_numpy()
    comble = origine.interpolate(method="time", limit_direction="both")
    comble = comble.mask(manquant & (tailles > max_pas))
    mesures = np.flatnonzero(~manquant)          # pas d'extrapolation hors plage mesurée
    if mesures.size:
        comble.iloc[:mesures[0]] = origine.iloc[:mesures[0]]
        comble.iloc[mesures[-1] + 1:] = origine.iloc[mesures[-1] + 1:]
    full_data[col] = comble
    full_data[f"Statut_{col}"] = np.where(
        ~manquant, "Mesurée", np.where(comble.notna().to_numpy(), "Interpolée", "Manquante"))

full_data["Niveau_(mNGF)"] = cote_ngf(full_data["Niveau_(cm)"])
full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]
print(f"Zéro de l'échelle à {cote_ngf(0):.4f} m NGF")
display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in PARAMETRES}).fillna(0).astype(int).T)

## 14. Sauvegarde et graphe de synthèse

Un paramètre par grandeur, avec son statut et la sonde qui l'a fourni. Le détail
capteur par capteur reste dans le fichier écrit en cellule 8.

In [ ]:
finaux = PARAMETRES + ["Niveau_(mNGF)"]
colonnes = [c for p in finaux for c in (p, f"Statut_{p}", f"{p}_source") if c in full_data]
full_data[colonnes].to_excel(SORTIE_FINALE)
print(f"{SORTIE_FINALE} : {len(full_data)} pas x {len(colonnes)} colonnes")

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, gridspec_kw={"hspace": 0.05})
axes[0].plot(full_data.index, full_data["Niveau_(cm)"].rolling(12, center=True).mean(),
             color="lightseagreen")
axes[0].set_ylabel("Niveau (cm)", color="lightseagreen")
if os.path.exists(PLUIE_PATH):
    pluie = pd.read_csv(PLUIE_PATH)
    ax = axes[0].twinx()
    ax.bar(pd.to_datetime(pluie["Date"], errors="coerce"), pluie["Precipitation (mm)"],
           width=0.8, color="royalblue")
    ax.invert_yaxis()
    ax.set_ylabel("Précipitations (mm)", color="royalblue")

axes[1].plot(full_data.index, full_data["Conductivité_Moyenne_Mobile"], color="black")
axes[1].set_ylabel("Conductivité (µS/cm)")
ax = axes[1].twinx()
ax.plot(full_data.index, full_data["Température"].rolling(12, center=True).mean(), color="crimson")
ax.set_ylabel("Température (°C)", color="crimson")

axes[2].plot(full_data.index, full_data["Turbidité_(NTU)"].rolling(24, center=True).mean(),
             color="darkorange")
axes[2].set_ylabel("Turbidité (NTU)", color="darkorange")
axes[2].set_ylim(0, 100)
ax = axes[2].twinx()
ax.plot(full_data.index, full_data["O2_(mg/l)"].rolling(24, center=True).mean(),
        color="darkmagenta")
ax.set_ylabel("Oxygène (mg/L)", color="darkmagenta")

plt.savefig(SORTIE_SVG, format="svg")
plt.show()